In [1]:
import os
from dotenv import load_dotenv
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
# from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

True

In [32]:
user_prompt = "אילה פרטים צריך למלא בעת הרשמה לאפליקציה?"

In [3]:
# Ollama:
llm = ChatOllama(model="gemma3:12b")
embedding = OllamaEmbeddings(model="qwen3-embedding:8b")

In [4]:
from langchain_community.document_loaders import Docx2txtLoader

try:
    docx_loader = Docx2txtLoader("../DataIngestParsing/data/docx/terms_and_condition_young_app.docx")
    docx = docx_loader.load()

    print(f"Processed into {len(docx)} documents")
    print(f"Content {docx[0].page_content[:50]}...")
    print(f"Metadata {docx[0].metadata}")
except Exception as e:
    print(f"Error processing {e}")

Processed into 1 documents
Content תנאי שימוש

כללי:

1. מבוטח יקר, אנו שמחים שבחרת ל...
Metadata {'source': '../DataIngestParsing/data/docx/terms_and_condition_young_app.docx'}


In [5]:
import re

raw_text = docx[0].page_content
source = docx[0].metadata["source"]

# Split only on top-level numbered paragraphs (1. 2. 3.)
# NOT on sub-paragraphs like 3.1 3.2 — the (?!\d) prevents that
parts = re.split(r'(?=\n\d+\.(?!\d))', raw_text)

chunks = []
for part in parts:
    text = part.strip()
    if text:
        chunks.append(Document(page_content=text, metadata={"source": source}))

print(f"Created {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content[:150])


Created 65 chunks

--- Chunk 1 ---
תנאי שימוש

כללי:

--- Chunk 2 ---
1. מבוטח יקר, אנו שמחים שבחרת לעשות שימוש באפליקציית הרכב של תכנית הביטוח ״הפניקס צעיר״ (להלן: ״האפליקציה״, ו- "הפניקס" או "החברה") אשר דרכה ניתן לרכו

--- Chunk 3 ---
2. עם כניסתך לאפליקציה ובטרם תבצע פעולה כלשהי באמצעות האפליקציה, שימוש בשירות או במידע כלשהו הקיים באפליקציה, הנך מתבקש לקרוא בעיון ולאשר את תנאי השימ

--- Chunk 4 ---
3. תנאי השימוש ומדיניות הפרטיות להלן באים להוסיף על תנאי השימוש ומדיניות הפרטיות של אתר הפניקס
https://www.fnx.co.il/customer-service/terms-of-use/pri

--- Chunk 5 ---
4. ייתכן אפשרות שבמידע המוצג למבוטח באמצעות האפליקציה נפלו שיבושים ו/או שגיאות ו/או אי דיוקים. בכל מקרה של סתירה בין המידע שיתקבל במסגרת השירות לבין ה

--- Chunk 6 ---
5. הרישום והכניסה לאפליקציה, לרבות לרשימת הדיוור ו/או דרכי יצירת הקשר עם החברה וכן השימוש במידע שמסר המבוטח לחברה ו/או שהצטבר אודותיו בעת השימוש באפלי

--- Chunk 7 ---
6. נבקש לעדכנך כי בעת השימוש באפליקציה, עשויות להופיע פרסומות ומודעות המותאמות לך אישית, כ

In [6]:
# FAISS is an alternative simple vector store (memory only)
# dense_vectorstore = FAISS.from_documents(chunks, embedding=embedding)

dense_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./vector_store",
    collection_name="terms_conditions"
)

# print(f"Added {len(chunks)} chunks to vector store")
# print(f"Total documents in store: {vector_store._collection.count()}")


In [43]:
# Denes Retriever - vector/semantic search: understands meaning and paraphrases. "Does Phoenix share my data?" → finds chunk about third-party even if exact words differ.
dense_retriever = dense_vectorstore.as_retriever(search_kwargs={"k": 2})

# Sparse Retriever (BM25) - keyword frequency (classic IR): rewards exact term overlap. If the query contains "third-party", BM25 scores chunks containing that exact phrase higher.
sparse_retriever = BM25Retriever.from_documents(chunks)
sparse_retriever.k = 3

In [44]:
# The Combination of both give us the best results for embedding long text (unlike xlsx files)
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)

In [16]:
# Similarity TEST:

query = "הפניקס הולכת לשתף צדדים שלישיים עם המידע שנאסף מהאפליקציה?"

# similar_docs = vector_store.similarity_search(query, k=2)
similar_docs_with_scores = dense_vectorstore.similarity_search_with_score(query, k=2)

# scores with ChromaDB:
# Closer to 0 = Similar result
# Zero means identical vectors

# print(similar_docs[0])
# print("---------")
print(similar_docs_with_scores)

[(Document(metadata={'source': '../DataIngestParsing/data/docx/terms_and_condition_young_app.docx'}, page_content='28. הפניקס לא תשתף צדדים שלישיים, למעט איתוראן לשם תפעול שוטף של תכנית הביטוח, במידע המאפשר זיהוי אישי, אלא אם קיימת חובה לפי דין או אם הדבר מתבקש באופן מפורש על ידך או באופן אחר כמפורט בתנאי השימוש.'), 0.45322519540786743), (Document(metadata={'source': '../DataIngestParsing/data/docx/terms_and_condition_young_app.docx'}, page_content='4. הפניקס אוספת בעת עשיית שימוש באפליקציה נתונים, כמפורט להלן. בעשיית שימוש באפליקציה ובמסירת נתונים אישיים כלשהם הנך מסכים לכך שהפניקס תעשה שימוש בנתונים אלו בהתאם להצהרת פרטיות זו.'), 0.4783560037612915)]


### Reranking Hybrid Strategy
The goal of the reranking stage is to reorder the retrieved top_k results based on their actual relevance to the user’s query.

After the initial vector search retrieves candidate documents, an additional LLM (or reranker model) evaluates each result and assigns a more accurate relevance score. This second-pass ranking helps surface the most contextually relevant documents before generating the final response.

In [45]:
ranking_prompt = ChatPromptTemplate.from_template("""
    You are a helpful assistance. Your task is to rank the following documents from most to least relevant based on the user's query.

    User query: "{question}"

    Documents:
    {documents}

    Instructions:
    - Think about the relevance of each document to the user's question.
    - Return a list of documents indices in ranked order, starting from the most relevant.

    Output format: comma-separated document indices (e.g. 2,1,3,0,...)
""")

In [46]:
retrieved_docs = hybrid_retriever.invoke(user_prompt)
print(len(retrieved_docs))
print(retrieved_docs[0])

4
page_content='6. במסגרת הרישום לשירות תתבקש לספק מספר פרטים אישיים:

שם מלא

תעודת זהות של הנהג

תאריך הוצאת רישיון

תאריך לידה

כתובת הנהג

מספר טלפון

כתובת מייל

מגדר

מספר רישוי רכב/ים.

נתונים אלה ישמרו במערכות החברה וישמשו למתן שירות ומידע למבוטח.' metadata={'source': '../DataIngestParsing/data/docx/terms_and_condition_young_app.docx'}


In [47]:
doc_lines = [f"{i + 1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)
print(doc_lines[1])
print("-------")
formatted_docs

2. 5. הרישום והכניסה לאפליקציה, לרבות לרשימת הדיוור ו/או דרכי יצירת הקשר עם החברה וכן השימוש במידע שמסר המבוטח לחברה ו/או שהצטבר אודותיו בעת השימוש באפליקציה ו/או התקבל מצד ג' לאור הרשאת המבוטח (כמו למשל, מידע מאתרי מידע ביטוחי כ"הר הביטוח"), יתבצע בהתאם להוראות הדין ומדיניות הפרטיות להלן, המהווה חלק בלתי נפרד מתנאי השימוש.
-------


'1. 6. במסגרת הרישום לשירות תתבקש לספק מספר פרטים אישיים:\n\nשם מלא\n\nתעודת זהות של הנהג\n\nתאריך הוצאת רישיון\n\nתאריך לידה\n\nכתובת הנהג\n\nמספר טלפון\n\nכתובת מייל\n\nמגדר\n\nמספר רישוי רכב/ים.\n\nנתונים אלה ישמרו במערכות החברה וישמשו למתן שירות ומידע למבוטח.\n2. 5. הרישום והכניסה לאפליקציה, לרבות לרשימת הדיוור ו/או דרכי יצירת הקשר עם החברה וכן השימוש במידע שמסר המבוטח לחברה ו/או שהצטבר אודותיו בעת השימוש באפליקציה ו/או התקבל מצד ג\' לאור הרשאת המבוטח (כמו למשל, מידע מאתרי מידע ביטוחי כ"הר הביטוח"), יתבצע בהתאם להוראות הדין ומדיניות הפרטיות להלן, המהווה חלק בלתי נפרד מתנאי השימוש.\n3. 15. מובהר, כי עליך להקפיד על מסירת פרטים מדויקים ונכונים, על-מנת שהשימוש במערכת ו/או באפליקציה יתאפשר במהירות וללא תקלות. להסרת ספק, הקלדת פרטים אישיים כוזבים אסורה לחלוטין, מהווה עוולה אזרחית ואף עבירה פלילית, והעושה כן צפוי להליכים משפטיים, פליליים ואזרחיים, לרבות תביעות נזיקין בגין נזקים שיגרמו להפניקס ו/או לכל מי מטעמה עקב כך. עוד מובהר כי מתן פרטים שאינם מלאים וכנים יש בו כדי להשפיע על תשלום תגמו

In [48]:
chain = ranking_prompt | llm |  StrOutputParser()

In [49]:
ranking_response = chain.invoke({
    "question": user_prompt,
    "documents": formatted_docs
})

ranking_response

'1, 6, 2, 3, 4'

In [50]:
# Parse and rerank
indices = [int(x.strip()) - 1 for x in ranking_response.split(",") if x.strip().isdigit()]
print(indices)
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]
print(reranked_docs)

[0, 5, 1, 2, 3]
[Document(metadata={'source': '../DataIngestParsing/data/docx/terms_and_condition_young_app.docx'}, page_content='6. במסגרת הרישום לשירות תתבקש לספק מספר פרטים אישיים:\n\nשם מלא\n\nתעודת זהות של הנהג\n\nתאריך הוצאת רישיון\n\nתאריך לידה\n\nכתובת הנהג\n\nמספר טלפון\n\nכתובת מייל\n\nמגדר\n\nמספר רישוי רכב/ים.\n\nנתונים אלה ישמרו במערכות החברה וישמשו למתן שירות ומידע למבוטח.'), Document(metadata={'source': '../DataIngestParsing/data/docx/terms_and_condition_young_app.docx'}, page_content='5. הרישום והכניסה לאפליקציה, לרבות לרשימת הדיוור ו/או דרכי יצירת הקשר עם החברה וכן השימוש במידע שמסר המבוטח לחברה ו/או שהצטבר אודותיו בעת השימוש באפליקציה ו/או התקבל מצד ג\' לאור הרשאת המבוטח (כמו למשל, מידע מאתרי מידע ביטוחי כ"הר הביטוח"), יתבצע בהתאם להוראות הדין ומדיניות הפרטיות להלן, המהווה חלק בלתי נפרד מתנאי השימוש.'), Document(metadata={'source': '../DataIngestParsing/data/docx/terms_and_condition_young_app.docx'}, page_content='15. מובהר, כי עליך להקפיד על מסירת פרטים מדויקים ונכונ

#### f

In [51]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a professional customer service representative for Phoenix Insurance Company.
        ***Always respond in Hebrew regardless of the language used in the question***.

        Your goal is to provide accurate information about the company's terms and condition.

        Guidelines:
        * Use a professional, patient, and respectful tone.
        * If information is missing or not found in the system — state it clearly.
        * Do not fabricate information that does not exist in the provided data.
        * If the question is unrelated to terms and condition data — politely respond that you cannot assist with that topic.
        * Phrase answers in a natural, human way — not as a raw technical data list.

        ***Always respond in Hebrew***.

        Context: {context}
    """),
    ("human", "{input}")
])

In [52]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm, system_prompt)
rag_chain = create_retrieval_chain(hybrid_retriever, document_chain)


In [53]:
response = rag_chain.invoke({
    "input": user_prompt,
})

print(response.get("answer"))

שלום רב,

בעת הרישום לשירות האפליקציה, תתבקש/י לספק את הפרטים הבאים:

*   שם מלא
*   תעודת זהות של הנהג
*   תאריך הוצאת רישיון
*   תאריך לידה
*   כתובת הנהג
*   מספר טלפון
*   כתובת מייל
*   מגדר
*   מספר רישוי רכב/ים.

חשוב מאוד למלא את כל הפרטים בצורה מדויקת ונכונה כדי להבטיח שהשימוש באפליקציה יתאפשר בצורה חלקה וללא תקלות. מתן פרטים כוזבים או לא שלמים עלול להשפיע על תשלום תגמולי הביטוח ולגרור השלכות משפטיות.

במידה ויש לך שאלות נוספות, אשמח לעזור.
